# EasyOCR speed benchmark

Runs many **random 10-frame sequences**: each trial either keeps all base frames or swaps **one random slot** for the numbered image — mimicking conditional jersey OCR in the pipeline.

Assets: `images/base/` (10, no number) and `images/with_number/` (1). See [README.md](README.md).

In [ ]:
from pathlib import Path

EXPERIMENT_DIR = Path.cwd()
BASE_DIR = EXPERIMENT_DIR / "images" / "base"
NUMBER_DIR = EXPERIMENT_DIR / "images" / "with_number"
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
SEQUENCE_LENGTH = 10

LANGUAGES = ["en"]
USE_GPU = None  # None = auto; False = force CPU
WARMUP = True

# Fraction of trials that inject the numbered frame at a random slot
P_HAS_NUMBER = 0.3
N_TRIALS = 50
RANDOM_SEED = 42

In [ ]:
import random
import re
import time
from statistics import mean, stdev

import easyocr
import pandas as pd
from PIL import Image

DIGIT_RE = re.compile(r"\d")

In [ ]:
def list_images(directory: Path) -> list[Path]:
    paths = [
        p
        for p in directory.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    ]
    return sorted(paths, key=lambda p: p.name.lower())


base_paths = list_images(BASE_DIR)
number_paths = list_images(NUMBER_DIR)

if len(base_paths) != SEQUENCE_LENGTH:
    raise FileNotFoundError(
        f"Expected {SEQUENCE_LENGTH} images in {BASE_DIR}, found {len(base_paths)}"
    )
if len(number_paths) != 1:
    raise FileNotFoundError(
        f"Expected exactly 1 image in {NUMBER_DIR}, found {len(number_paths)}"
    )

number_path = number_paths[0]
print(f"Base frames ({len(base_paths)}):")
for i, p in enumerate(base_paths):
    with Image.open(p) as im:
        print(f"  [{i}] {p.name}  {im.size[0]}x{im.size[1]}")
with Image.open(number_path) as im:
    print(f"Number frame: {number_path.name}  {im.size[0]}x{im.size[1]}")

In [ ]:
gpu = USE_GPU
if gpu is None:
    try:
        import torch

        gpu = torch.cuda.is_available()
    except ImportError:
        gpu = False

print(f"Initializing EasyOCR reader (gpu={gpu}, languages={LANGUAGES})...")
t0 = time.perf_counter()
reader = easyocr.Reader(LANGUAGES, gpu=gpu, verbose=False)
init_s = time.perf_counter() - t0
print(f"Reader ready in {init_s:.2f} s")

In [ ]:
def run_ocr(path: Path) -> list[tuple[str, float]]:
    raw = reader.readtext(str(path), detail=1, paragraph=False)
    return [(str(item[1]).strip(), float(item[2])) for item in raw]


def texts_joined(detections: list[tuple[str, float]]) -> str:
    return " | ".join(t for t, _ in detections if t)


def has_digit(detections: list[tuple[str, float]]) -> bool:
    return any(DIGIT_RE.search(t) for t, _ in detections)


def build_sequence(rng) -> tuple[list[Path], bool, int | None]:
    """Return (frame paths, injected?, slot index or None)."""
    slots = list(base_paths)
    if rng.random() >= P_HAS_NUMBER:
        return slots, False, None
    idx = rng.randrange(SEQUENCE_LENGTH)
    slots[idx] = number_path
    return slots, True, idx

In [ ]:
if WARMUP:
    print(f"Warmup on {base_paths[0].name}...")
    _ = run_ocr(base_paths[0])
    print("Warmup done.")

In [ ]:
rng = random.Random(RANDOM_SEED)
rows: list[dict] = []

for trial in range(N_TRIALS):
    sequence, injected, inject_slot = build_sequence(rng)
    trial_start = time.perf_counter()
    for slot, path in enumerate(sequence):
        t0 = time.perf_counter()
        detections = run_ocr(path)
        elapsed_ms = (time.perf_counter() - t0) * 1000.0
        is_number_slot = injected and slot == inject_slot
        rows.append(
            {
                "trial": trial,
                "slot": slot,
                "injected": injected,
                "inject_slot": inject_slot,
                "is_number_slot": is_number_slot,
                "image": path.name,
                "elapsed_ms": round(elapsed_ms, 2),
                "text": texts_joined(detections) or "(none)",
                "has_digit": has_digit(detections),
                "n_boxes": len(detections),
            }
        )
    trial_ms = (time.perf_counter() - trial_start) * 1000.0
    for r in rows:
        if r["trial"] == trial:
            r["trial_total_ms"] = round(trial_ms, 2)

df = pd.DataFrame(rows)
print(f"Ran {N_TRIALS} trials × {SEQUENCE_LENGTH} frames = {len(df)} OCR calls")
df.head(12)

In [ ]:
trials = df.groupby("trial", as_index=False).first()
n_injected = int(trials["injected"].sum())
trial_totals = df.groupby("trial")["trial_total_ms"].first()

injected_trials = df[df["injected"]].drop_duplicates("trial")
hits = 0
for _, t in injected_trials.iterrows():
    slot = int(t["inject_slot"])
    row = df[(df["trial"] == t["trial"]) & (df["slot"] == slot)].iloc[0]
    if row["has_digit"]:
        hits += 1

times = df["elapsed_ms"].tolist()
times_no_number = df[~df["is_number_slot"]]["elapsed_ms"].tolist()
times_number_slot = df[df["is_number_slot"]]["elapsed_ms"].tolist()

print("--- configuration ---")
print(f"P_HAS_NUMBER:    {P_HAS_NUMBER}")
print(f"N_TRIALS:        {N_TRIALS}")
print(f"RANDOM_SEED:     {RANDOM_SEED}")
print(f"Reader init:     {init_s:.2f} s")
print(f"GPU:             {gpu}")
print(f"Warmup:          {WARMUP}")
print()
print("--- trial mix (representative sampling) ---")
print(f"Trials with number injected: {n_injected} / {N_TRIALS} ({100 * n_injected / N_TRIALS:.0f}%)")
print(f"Trials with no number:       {N_TRIALS - n_injected} / {N_TRIALS}")
if n_injected:
    print(f"OCR digit hit on inject slot: {hits} / {n_injected}")
print()
print("--- per-frame latency (all OCR calls) ---")
print(f"Mean:            {mean(times):.1f} ms")
if len(times) > 1:
    print(f"Std dev:         {stdev(times):.1f} ms")
print(f"Min / max:       {min(times):.1f} / {max(times):.1f} ms")
print(f"Effective FPS:   {1000.0 / mean(times):.2f}")
if times_no_number:
    print(f"Mean (no-number slots): {mean(times_no_number):.1f} ms  (n={len(times_no_number)})")
if times_number_slot:
    print(f"Mean (number slot):     {mean(times_number_slot):.1f} ms  (n={len(times_number_slot)})")
print()
print("--- per-sequence (10 frames, random placement) ---")
print(f"Mean batch time: {mean(trial_totals):.1f} ms  ({mean(trial_totals) / 1000:.2f} s)")
if len(trial_totals) > 1:
    print(f"Std dev:         {stdev(trial_totals):.1f} ms")
print(f"Batch FPS:       {10000.0 / mean(trial_totals):.2f}  (10 frames / mean batch)")

In [ ]:
# Inspect one injected trial and one no-number trial
sample_injected = trials.loc[trials["injected"], "trial"]
sample_plain = trials.loc[~trials["injected"], "trial"]

if len(sample_injected):
    t = int(sample_injected.iloc[0])
    sub = df[df["trial"] == t][["slot", "image", "is_number_slot", "elapsed_ms", "has_digit", "text"]]
    print(f"Example injected trial {t} (slot {int(sub.loc[sub['is_number_slot'], 'slot'].iloc[0])} is numbered):")
    display(sub)

if len(sample_plain):
    t = int(sample_plain.iloc[0])
    sub = df[df["trial"] == t][["slot", "image", "elapsed_ms", "has_digit", "text"]]
    print(f"Example no-number trial {t}:")
    display(sub)